# GRAIL — Interactive Module Exploration

This notebook walks through every layer of the GRAIL library, loading real
artifacts from the quickstart project and inspecting their structures.

It mirrors the workflow from the legacy `Nirvana GraphRag Search.py` script
but uses the clean GRAIL public API throughout.

**Prerequisites:**
- The quickstart project has been indexed: `grail index examples/quickstart`
- `DEEPINFRA_API_KEY` is set (via `.env` or environment)

**Sections:**
1. Setup & Config
2. Load Artifacts (parquets + mapping)
3. Inspect Documents, Text Units, Entities, Relationships
4. Inspect Communities & Reports
5. Graph Structure (NetworkX)
6. Retrieval Primitives (entity mapping, context building)
7. Local Search
8. Global Search
9. Document Search
10. Provenance Tracing (source → file chain)
11. Cost Tracking

## 1. Setup & Config

In [ ]:
import sys, os, json
from pathlib import Path

# Point at the repo root so `import grail` works even without pip install.
REPO_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

# Auto-load .env from the quickstart project.
from dotenv import load_dotenv
QUICKSTART = REPO_ROOT / "examples" / "quickstart"
load_dotenv(QUICKSTART / ".env")

import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print(f"Repo root: {REPO_ROOT}")
print(f"Quickstart: {QUICKSTART}")
print(f"DEEPINFRA_API_KEY set: {'DEEPINFRA_API_KEY' in os.environ}")

In [ ]:
from grail import GRAIL, Config, SearchResult, load_config
from grail.config import MANDATORY_ENTITY_TYPES
from grail.query import (
    AgentSearch, DocumentSearch, GlobalSearch, LocalSearch,
    load_artifacts_for_search, map_query_to_entities,
)
from grail.query.retrieval import (
    SearchArtifacts,
    build_community_context,
    build_entity_context,
    build_relationship_context,
    build_text_unit_context,
)
from grail.storage import LocalStorage

# Load config and build the GRAIL instance.
config = load_config(QUICKSTART / "grail.yaml")
grail = GRAIL.from_config(config)

print(f"Project:    {config.project_name}")
print(f"LLM:        {config.llm.endpoint} | {config.llm.model}")
print(f"Embeddings: {config.embeddings.endpoint} | {config.embeddings.model}")
print(f"Entity types: {config.indexing.entity_types}")
print(f"Mandatory injected: {MANDATORY_ENTITY_TYPES}")

## 2. Load Artifacts

Load all parquet artefacts and mapping.json produced by `grail index`.

In [ ]:
# Load using the same function the search layer uses internally.
artifacts = load_artifacts_for_search(grail.storage, grail._output_folder())

print("Artifact shapes:")
print(f"  documents:         {artifacts.documents.shape}")
print(f"  text_units:        {artifacts.text_units.shape}")
print(f"  entities:          {artifacts.entities.shape}")
print(f"  relationships:     {artifacts.relationships.shape}")
print(f"  nodes:             {artifacts.nodes.shape}")
print(f"  communities:       {artifacts.communities.shape}")
print(f"  community_reports: {artifacts.community_reports.shape}")
print(f"  mapping entries:   {len(artifacts.mapping)}")

## 3. Inspect Documents, Text Units, Entities, Relationships

### 3.1 Documents (`final_docs.parquet`)

In [ ]:
docs_df = artifacts.documents.copy()
print(f"Columns: {list(docs_df.columns)}")
print(f"Rows: {len(docs_df)}\n")

# Truncate raw_content for display.
display_df = docs_df.copy()
display_df["raw_content"] = display_df["raw_content"].str[:200] + "..."
display_df

### 3.2 Text Units (`final_text_units.parquet`)

In [ ]:
tu_df = artifacts.text_units.copy()
print(f"Columns: {list(tu_df.columns)}")
print(f"Rows: {len(tu_df)}")
print(f"Token range: {tu_df['n_tokens'].min()} - {tu_df['n_tokens'].max()}")
print(f"Total tokens: {tu_df['n_tokens'].sum()}")

# Show first few, truncating text.
display_tu = tu_df.copy()
display_tu["text"] = display_tu["text"].str[:150] + "..."
display_tu[["id", "n_tokens", "document_ids", "entity_ids", "text"]].head(5)

### 3.3 Entities (`final_entities.parquet`)

In [ ]:
ent_df = artifacts.entities.copy()
print(f"Columns: {list(ent_df.columns)}")
print(f"Rows: {len(ent_df)}")
print(f"\nEntity types distribution:")
print(ent_df["type"].value_counts().to_string())

# Top entities by degree (most connected).
print(f"\nTop 15 entities by degree:")
ent_df.nlargest(15, "degree")[["name", "type", "degree", "description"]].head(15)

In [ ]:
# Embedding coverage: what percentage of entities have description embeddings?
has_emb = ent_df["description_embedding"].apply(
    lambda x: x is not None and hasattr(x, "__len__") and len(x) > 0
)
print(f"Entities with embeddings: {has_emb.sum()}/{len(ent_df)} ({has_emb.mean()*100:.1f}%)")

# Show embedding dimensionality.
sample_emb = ent_df.loc[has_emb, "description_embedding"].iloc[0]
print(f"Embedding dimension: {len(sample_emb)}")

### 3.4 Relationships (`final_relationships.parquet`)

In [ ]:
rel_df = artifacts.relationships.copy()
print(f"Columns: {list(rel_df.columns)}")
print(f"Rows: {len(rel_df)}")
print(f"Weight range: {rel_df['weight'].min():.2f} - {rel_df['weight'].max():.2f}")
print(f"Rank range: {rel_df['rank'].min():.1f} - {rel_df['rank'].max():.1f}")

# Top relationships by rank.
print(f"\nTop 15 relationships by rank:")
top_rels = rel_df.nlargest(15, "rank")[["source", "target", "weight", "rank", "description"]]
top_rels["description"] = top_rels["description"].str[:100]
top_rels

## 4. Communities & Reports

### 4.1 Nodes & Community Structure (`final_nodes.parquet`)

In [ ]:
nodes_df = artifacts.nodes.copy()
print(f"Columns: {list(nodes_df.columns)}")
print(f"Rows: {len(nodes_df)}")
print(f"\nCommunity levels: {sorted(nodes_df['level'].unique())}")
print(f"\nCommunities per level:")
for lvl in sorted(nodes_df["level"].unique()):
    subset = nodes_df[nodes_df["level"] == lvl]
    n_communities = subset["community"].nunique()
    n_nodes = len(subset)
    print(f"  Level {lvl}: {n_communities} communities, {n_nodes} node assignments")

### 4.2 Communities (`final_communities.parquet`)

In [ ]:
comm_df = artifacts.communities.copy()
print(f"Columns: {list(comm_df.columns)}")
print(f"Rows: {len(comm_df)}")
comm_df.head(10)

### 4.3 Community Reports (`final_community_reports.parquet`)

These are the LLM-generated narrative summaries of each community. The legacy
script used `full_content` and `full_content_json` columns.

In [ ]:
reports_df = artifacts.community_reports.copy()
print(f"Columns: {list(reports_df.columns)}")
print(f"Rows: {len(reports_df)}")

# Show a sample report.
if not reports_df.empty:
    sample = reports_df.iloc[0]
    print(f"\n--- Sample Report (community={sample.get('community', '?')}) ---")
    for col in ["title", "summary", "rank"]:
        if col in reports_df.columns:
            val = sample.get(col, "")
            print(f"  {col}: {str(val)[:300]}")
    if "full_content" in reports_df.columns:
        print(f"  full_content (first 500 chars):")
        print(f"    {str(sample['full_content'])[:500]}")

### 4.4 Mapping.json

The citation root — maps document IDs to original file paths.

In [ ]:
mapping = artifacts.mapping
print(f"Mapping entries: {len(mapping)}\n")
for doc_id, info in mapping.items():
    print(f"  {doc_id}:")
    for k, v in info.items():
        display_v = str(v)[:200] if isinstance(v, str) else v
        print(f"    {k}: {display_v}")

## 5. Graph Structure (NetworkX)

Load the GraphML file and inspect the knowledge graph topology.

In [ ]:
out = grail._output_folder()
graphml_key = f"{out}/entity_relationship_graph.graphml"

with grail.storage.open_for_read(graphml_key) as path:
    G = nx.read_graphml(path)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")

G_undirected = G.to_undirected()
print(f"Connected components: {nx.number_connected_components(G_undirected)}")

# Largest connected component.
lcc = max(nx.connected_components(G_undirected), key=len)
print(f"Largest component: {len(lcc)} nodes")

# Top 10 nodes by degree centrality.
degree_cent = nx.degree_centrality(G)
top_central = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:10]
print(f"\nTop 10 by degree centrality:")
for node, cent in top_central:
    print(f"  {node}: {cent:.4f}")

## 6. Retrieval Primitives

These are the building blocks that local/global search compose internally.
Running them directly lets you inspect exactly what goes into the LLM context.

In [ ]:
# 6.1 Map a query to the top-k entities via embedding similarity.
query_text = "cancer cachexia treatment guidelines"
query_embedding = await grail.embeddings.embed_one(query_text)
print(f"Query embedding dimension: {len(query_embedding)}")

ranked_entities = map_query_to_entities(
    query_embedding=query_embedding,
    entities_df=artifacts.entities,
    top_k=10,
)
print(f"\nTop 10 entities for '{query_text}':")
ranked_entities[["name", "type", "degree", "description"]].head(10)

In [ ]:
# 6.2 Build entity context — the CSV-like block that goes into the prompt.
entity_text, entity_rows = build_entity_context(ranked_entities, max_tokens=2000)
print(f"Entity context: {len(entity_text)} chars, {len(entity_rows)} entities used")
print(f"\n--- Entity context block (first 800 chars) ---\n{entity_text[:800]}")

In [ ]:
# 6.3 Build relationship context for the selected entities.
selected_names = entity_rows["name"].tolist()
rel_text, rel_rows = build_relationship_context(
    artifacts.relationships, selected_names, max_tokens=2000
)
print(f"Relationship context: {len(rel_text)} chars, {len(rel_rows)} relationships used")
print(f"\n--- Relationship context block (first 800 chars) ---\n{rel_text[:800]}")

In [ ]:
# 6.4 Build community context.
comm_text, comm_rows = build_community_context(artifacts.community_reports, max_tokens=4000)
text = comm_text if isinstance(comm_text, str) else "\n".join(comm_text)
print(f"Community context: {len(text)} chars, {len(comm_rows)} reports used")
print(f"\n--- Community context block (first 800 chars) ---\n{text[:800]}")

In [ ]:
# 6.5 Build text unit context — source text chunks mentioning selected entities.
source_text, source_rows = build_text_unit_context(
    artifacts.text_units, selected_names, max_tokens=3000,
    documents=artifacts.documents, mapping=artifacts.mapping,
)
print(f"Text unit context: {len(source_text)} chars, {len(source_rows)} units used")
print(f"\n--- Text unit context block (first 800 chars) ---\n{source_text[:800]}")

## 7. Local Search

The main search mode — maps query to entities, builds multi-source context,
sends to LLM. Equivalent to the legacy `LocalSearchNirvana.asearch()`.

In [ ]:
local_result = await grail.search(
    "What are the main treatments for cancer cachexia?",
    mode="local",
)

print(f"Completion time: {local_result.completion_time:.2f}s")
print(f"LLM calls: {local_result.llm_calls}")
print(f"Context data keys: {list(local_result.context_data.keys()) if isinstance(local_result.context_data, dict) else 'N/A'}")
print(f"\n{'='*80}")
print("RESPONSE:")
print(f"{'='*80}")
print(local_result.response)

In [ ]:
# Inspect the context_data DataFrames — same structure as legacy result.context_data.
if isinstance(local_result.context_data, dict):
    for key, df in local_result.context_data.items():
        if isinstance(df, pd.DataFrame):
            print(f"\n--- {key} ---")
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)}")
            display_df = df.copy()
            for col in display_df.columns:
                if display_df[col].dtype == object:
                    display_df[col] = display_df[col].astype(str).str[:80]
            display(display_df.head(10))

### 7.1 Local Search with Conversation History

Test context-aware follow-up queries.

In [ ]:
history = [
    {"role": "user", "content": "Tell me about gliomas."},
    {"role": "assistant", "content": "Gliomas are the most common primary brain tumors..."},
]

followup_result = await grail.search(
    "What treatments are recommended?",
    mode="local",
    conversation_history=history,
)

print(f"Completion time: {followup_result.completion_time:.2f}s")
print(f"\nRESPONSE:\n{followup_result.response[:500]}")

### 7.2 Local Search with Entity Filters

Force-include or exclude specific entities.

In [ ]:
filtered_result = await grail.search(
    "What are the guidelines?",
    mode="local",
    include_entity_names=["SEOM"],
)

print(f"Completion time: {filtered_result.completion_time:.2f}s")
print(f"\nEntities used:")
if isinstance(filtered_result.context_data, dict) and "entities" in filtered_result.context_data:
    ents = filtered_result.context_data["entities"]
    if isinstance(ents, pd.DataFrame) and not ents.empty:
        print(ents[["name", "type"]].to_string(index=False))
print(f"\nRESPONSE:\n{filtered_result.response[:500]}")

## 8. Global Search

Synthesizes answers from community reports. Equivalent to the legacy
`GlobalSearchNirvana.asearch()`. Uses single-pass reduce when context
fits in one chunk, map-reduce otherwise.

In [ ]:
global_result = await grail.search(
    "What are the main themes covered in the indexed documents?",
    mode="global",
)

print(f"Completion time: {global_result.completion_time:.2f}s")
print(f"LLM calls: {global_result.llm_calls}")
print(f"Context data keys: {list(global_result.context_data.keys()) if isinstance(global_result.context_data, dict) else 'N/A'}")
print(f"\n{'='*80}")
print("RESPONSE:")
print(f"{'='*80}")
print(global_result.response)

In [ ]:
# Inspect which community reports were used in global search.
if isinstance(global_result.context_data, dict) and "reports" in global_result.context_data:
    used_reports = global_result.context_data["reports"]
    if isinstance(used_reports, pd.DataFrame) and not used_reports.empty:
        print(f"Reports used: {len(used_reports)}")
        cols_to_show = [c for c in ["community", "title", "rank", "summary"] if c in used_reports.columns]
        display_r = used_reports[cols_to_show].copy()
        if "summary" in display_r.columns:
            display_r["summary"] = display_r["summary"].astype(str).str[:100]
        display(display_r.head(10))

## 9. Document Search

Scoped search within a single source document — useful when the user
asks about a specific file.

In [ ]:
# List available documents so you can pick one.
print("Indexed documents:")
for _, row in artifacts.documents.iterrows():
    print(f"  - {row['title']} (id={row['id'][:8]}...)")

In [ ]:
doc_result = await grail.search(
    "What does this document cover?",
    mode="document",
    document="cachexia",  # matches by title substring
)

print(f"Completion time: {doc_result.completion_time:.2f}s")
print(f"LLM calls: {doc_result.llm_calls}")
print(f"Context data keys: {list(doc_result.context_data.keys()) if isinstance(doc_result.context_data, dict) else 'N/A'}")
print(f"\n{'='*80}")
print("RESPONSE:")
print(f"{'='*80}")
print(doc_result.response)

## 10. Provenance Tracing

Reproduce the legacy script's source-document chain:
`search result → sources (text units) → document_ids → mapping → original files`

In [ ]:
# Use the local_result from section 7.
ctx = local_result.context_data
assert isinstance(ctx, dict), "context_data should be a dict"

print("Context data keys:", list(ctx.keys()))
print()

# Trace source documents from the "sources" DataFrame.
if "sources" in ctx and isinstance(ctx["sources"], pd.DataFrame) and not ctx["sources"].empty:
    sources_df = ctx["sources"]
    print(f"Source text units in context: {len(sources_df)}")
    
    # Collect all document IDs referenced by source text units.
    cited_doc_ids = set()
    for _, row in sources_df.iterrows():
        dids = row.get("document_ids")
        if isinstance(dids, list):
            cited_doc_ids.update(dids)
        elif isinstance(dids, str):
            cited_doc_ids.add(dids)
    
    print(f"\nCited document IDs: {cited_doc_ids}")
    
    # Resolve to file names via the documents parquet and mapping.json.
    cited_files = []
    for doc_id in cited_doc_ids:
        match = artifacts.documents[artifacts.documents["id"] == doc_id]
        if not match.empty:
            cited_files.append(match.iloc[0]["title"])
        elif doc_id in artifacts.mapping:
            cited_files.append(artifacts.mapping[doc_id].get("original_path", doc_id))
        else:
            cited_files.append(f"[unresolved: {doc_id}]")
    
    print(f"\nResolved source files:")
    for f in cited_files:
        print(f"  - {f}")
else:
    print("No source text units in context (index may be too small).")

## 11. Cost Tracking

Review token usage and cost estimates accumulated during this session.

In [ ]:
tracker = grail.cost_tracker

print("Cost summary by tag:")
summary = tracker.summary(by="tag")
for tag, info in summary.items():
    print(f"  {tag}: {info}")

print(f"\nTotal cost: {tracker.render_total_cost()}")
print(f"Pricing status: {tracker.pricing_status()}")

# Individual usage records.
if tracker.records:
    print(f"\nDetailed records ({len(tracker.records)} calls):")
    for r in tracker.records[:10]:
        print(f"  [{r.tag}] {r.model} — in:{r.prompt_tokens} out:{r.completion_tokens} cost:${r.cost_usd:.6f}")
    if len(tracker.records) > 10:
        print(f"  ... and {len(tracker.records) - 10} more")

## 12. GRAIL Status

Quick health check — shows which artefacts exist.

In [ ]:
status = grail.status()
print(f"Project: {status['project_name']}")
print(f"Storage: {status['storage']}")
print(f"\nArtefacts:")
for name, exists in status["artefacts"].items():
    mark = "OK" if exists else "MISSING"
    print(f"  {name:20s} {mark}")

## 13. SearchResult Structure Reference

Quick reference for the `SearchResult` dataclass returned by all search methods.

```python
@dataclass
class SearchResult:
    response: str | dict | list        # The LLM's answer
    context_data: dict[str, DataFrame]  # DataFrames keyed by: entities, relationships, reports, sources
    context_text: str                   # The raw text block sent to the LLM
    completion_time: float              # Wall-clock seconds
    llm_calls: int                      # Number of LLM API calls made
```

**`context_data` keys by search mode:**

| Mode | Keys |
|------|------|
| `local` | `entities`, `relationships`, `reports`, `sources` |
| `global` | `reports` (+ `map_points` if map-reduce was used) |
| `document` | `entities`, `relationships`, `sources` |
| `agent` | `{tool}_{key}` for each tool call (e.g. `local_search_entities`) |